# Emotion Classification with Attention

## 0. Setup & Installs

In [ ]:
!pip install -q datasets evaluate transformers accelerate contractions


In [ ]:
import os, re, json, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, GRU, Bidirectional, Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))


## Stage 1 — Dataset Preparation
We use the **GoEmotions** dataset (simplified config: single-label, 27 emotions + neutral).
Source: https://huggingface.co/datasets/google-research-datasets/go_emotions


In [ ]:
from datasets import load_dataset

raw = load_dataset('google-research-datasets/go_emotions', 'simplified')
print(raw)
label_names = raw['train'].features['labels'].feature.names
print(len(label_names), 'raw labels:', label_names)


In [ ]:
train_df = raw['train'].to_pandas()
val_df   = raw['validation'].to_pandas()
test_df  = raw['test'].to_pandas()

for df in (train_df, val_df, test_df):
    df['labels'] = df['labels'].apply(list)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()


In [ ]:
print(train_df.isna().sum())
print('Empty text rows:', (train_df['text'].str.strip() == '').sum())


### Map GoEmotions' 27 fine-grained labels -> 6 core emotions
GoEmotions labels don't map 1:1 to Ekman's 6 basic emotions, so we group the closest
fine-grained emotions under each core class and drop anything that doesn't fit well
(e.g. 'neutral', 'admiration', 'confusion', 'curiosity', 'realization', 'approval',
'disapproval', 'caring', 'desire', 'relief', 'embarrassment', 'pride', 'gratitude',
'optimism', 'nervousness' -> ambiguous / not one of the 6, dropped).
We also **drop multi-label rows that span more than one of our 6 target classes**
to keep this a clean single-label 6-class problem, as required by the brief.


In [ ]:
EMOTION_MAP = {
    'admiration': None, 'amusement': 'joy', 'anger': 'anger', 'annoyance': 'anger',
    'approval': None, 'caring': None, 'confusion': None, 'curiosity': None,
    'desire': None, 'disappointment': 'sadness', 'disapproval': None,
    'disgust': 'disgust', 'embarrassment': None, 'excitement': 'joy',
    'fear': 'fear', 'gratitude': 'joy', 'grief': 'sadness', 'joy': 'joy',
    'love': 'joy', 'nervousness': 'fear', 'optimism': None, 'pride': 'joy',
    'realization': None, 'relief': None, 'remorse': 'sadness', 'sadness': 'sadness',
    'surprise': 'surprise', 'neutral': None,
}
TARGET_EMOTIONS = ['joy', 'sadness', 'anger', 'fear', 'surprise', 'disgust']
label2id = {e: i for i, e in enumerate(TARGET_EMOTIONS)}
id2label = {i: e for e, i in label2id.items()}

def map_row_labels(label_ids):
    mapped = set()
    for lid in label_ids:
        name = label_names[lid]
        target = EMOTION_MAP.get(name)
        if target is not None:
            mapped.add(target)
    return mapped

def build_clean_df(df):
    mapped = df['labels'].apply(map_row_labels)
    keep = mapped.apply(lambda s: len(s) == 1)
    out = df[keep].copy()
    out['emotion'] = mapped[keep].apply(lambda s: next(iter(s)))
    out['label_id'] = out['emotion'].map(label2id)
    return out[['text', 'emotion', 'label_id']].reset_index(drop=True)

train_clean = build_clean_df(train_df)
val_clean   = build_clean_df(val_df)
test_clean  = build_clean_df(test_df)

print('Train/Val/Test sizes after mapping:', len(train_clean), len(val_clean), len(test_clean))
train_clean.head()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16,4))
for a, (name, d) in zip(ax, [('Train', train_clean), ('Val', val_clean), ('Test', test_clean)]):
    d['emotion'].value_counts().reindex(TARGET_EMOTIONS).plot(kind='bar', ax=a, color='steelblue')
    a.set_title(f'{name} class distribution')
    a.set_xlabel(''); a.set_ylabel('count')
plt.tight_layout(); plt.show()

train_clean['emotion'].value_counts()


## Stage 2 — Text Preprocessing
Lowercase, strip URLs/HTML/punctuation, normalize whitespace, tokenize, pad sequences.


In [ ]:
import contractions

URL_RE = re.compile(r'https?://\S+|www\.\S+')
HTML_RE = re.compile(r'<.*?>')
MULTI_SPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(' ', text)
    text = HTML_RE.sub(' ', text)
    text = contractions.fix(text)
    text = text.replace('[NAME]', ' ').replace('[RELIGION]', ' ')
    text = re.sub(r"[^a-z\s']", ' ', text)
    text = MULTI_SPACE_RE.sub(' ', text).strip()
    return text

for d in (train_clean, val_clean, test_clean):
    d['clean_text'] = d['text'].apply(clean_text)

train_clean = train_clean[train_clean['clean_text'].str.len() > 0].reset_index(drop=True)
val_clean   = val_clean[val_clean['clean_text'].str.len() > 0].reset_index(drop=True)
test_clean  = test_clean[test_clean['clean_text'].str.len() > 0].reset_index(drop=True)

train_clean[['text','clean_text','emotion']].head()


In [ ]:
MAX_VOCAB = 20000
MAX_LEN = 40

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train_clean['clean_text'])

def to_padded(df):
    seq = tokenizer.texts_to_sequences(df['clean_text'])
    return pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

X_train = to_padded(train_clean)
X_val   = to_padded(val_clean)
X_test  = to_padded(test_clean)

y_train = train_clean['label_id'].values
y_val   = val_clean['label_id'].values
y_test  = test_clean['label_id'].values

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print('Vocab size:', vocab_size)
print('X_train shape:', X_train.shape)

class_weights_arr = compute_class_weight('balanced', classes=np.arange(6), y=y_train)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}
print(class_weights)


## Stage 3 — Word Embeddings (GloVe)
We use **GloVe 100d** (`glove.6B.100d`), a strong general-purpose choice for short,
informal, single-sentence text like GoEmotions (Reddit comments) — good coverage,
fast to load, and a standard baseline for this kind of dataset.
On Kaggle, add the dataset **"glove6b100dtxt"** (Kaggle Datasets search) or download directly.


In [ ]:
GLOVE_CANDIDATES = [
    '/kaggle/input/glove6b100dtxt/glove.6B.100d.txt',
    '/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.100d.txt',
]
GLOVE_PATH = next((p for p in GLOVE_CANDIDATES if os.path.exists(p)), None)

if GLOVE_PATH is None:
    print('GloVe not found as Kaggle input dataset. Downloading...')
    os.system('wget -q http://nlp.stanford.edu/data/glove.6B.zip')
    os.system('unzip -q -o glove.6B.zip glove.6B.100d.txt')
    GLOVE_PATH = 'glove.6B.100d.txt'

print('Using GloVe file:', GLOVE_PATH)


In [ ]:
EMBED_DIM = 100
embeddings_index = {}
with open(GLOVE_PATH, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vec = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vec
print('Loaded', len(embeddings_index), 'word vectors')

embedding_matrix = np.zeros((vocab_size, EMBED_DIM))
hits = 0
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    vec = embeddings_index.get(word)
    if vec is not None:
        embedding_matrix[i] = vec
        hits += 1
print(f'Embedding coverage: {hits}/{vocab_size} ({hits/vocab_size:.1%})')


## Stage 4 — Build Deep Learning Models
Four architectures: LSTM, GRU, BiLSTM + Attention, and fine-tuned DistilBERT.


In [ ]:
def make_embedding_layer(trainable=False):
    return Embedding(input_dim=vocab_size, output_dim=EMBED_DIM,
                      weights=[embedding_matrix], input_length=MAX_LEN,
                      trainable=trainable, mask_zero=True)

def compile_model(model):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
    return model

EPOCHS = 12
BATCH_SIZE = 128
callbacks = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]


### Model 1 — LSTM

In [ ]:
lstm_model = Sequential([
    make_embedding_layer(),
    LSTM(128, dropout=0.3, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='softmax')
])
lstm_model = compile_model(lstm_model)
lstm_model.summary()


In [ ]:
history_lstm = lstm_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weights,
    callbacks=callbacks, verbose=1
)


### Model 2 — GRU

In [ ]:
gru_model = Sequential([
    make_embedding_layer(),
    GRU(128, dropout=0.3, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='softmax')
])
gru_model = compile_model(gru_model)
history_gru = gru_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weights,
    callbacks=callbacks, verbose=1
)


### Model 3 — BiLSTM + Attention
This is the most important custom model. We implement a Bahdanau-style additive
attention layer over the BiLSTM hidden states so we can later visualize which
words drove each prediction.


In [ ]:
class AttentionLayer(Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W = self.add_weight(name='att_W', shape=(input_shape[-1], self.units),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_b', shape=(self.units,),
                                  initializer='zeros', trainable=True)
        self.u = self.add_weight(name='att_u', shape=(self.units, 1),
                                  initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, hidden_states, mask=None):
        score = K.tanh(K.dot(hidden_states, self.W) + self.b)
        score = K.dot(score, self.u)
        score = K.squeeze(score, axis=-1)
        if mask is not None:
            score = tf.where(mask, score, -1e9 * tf.ones_like(score))
        weights = tf.nn.softmax(score, axis=1)
        weights_exp = tf.expand_dims(weights, axis=-1)
        context = tf.reduce_sum(hidden_states * weights_exp, axis=1)
        return context, weights

    def compute_mask(self, inputs, mask=None):
        return None

inp = Input(shape=(MAX_LEN,))
emb = make_embedding_layer()(inp)
bilstm_out = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2))(emb)
context, att_weights = AttentionLayer(64)(bilstm_out)
x = Dense(64, activation='relu')(context)
x = Dropout(0.3)(x)
out = Dense(6, activation='softmax')(x)

bilstm_att_model = Model(inputs=inp, outputs=out)
bilstm_att_model = compile_model(bilstm_att_model)
bilstm_att_model.summary()

bilstm_att_with_weights = Model(inputs=inp, outputs=[out, att_weights])


In [ ]:
history_bilstm_att = bilstm_att_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight=class_weights,
    callbacks=callbacks, verbose=1
)


### Model 4 — Fine-tuned DistilBERT (HuggingFace)
This will likely achieve the highest accuracy since it uses contextual pretraining.


In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
import evaluate

MODEL_NAME = 'distilbert-base-uncased'
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(df[['clean_text', 'label_id']].rename(columns={'clean_text':'text','label_id':'label'}))

hf_train = to_hf_dataset(train_clean)
hf_val   = to_hf_dataset(val_clean)
hf_test  = to_hf_dataset(test_clean)

def tokenize_fn(batch):
    return hf_tokenizer(batch['text'], truncation=True, max_length=64)

hf_train = hf_train.map(tokenize_fn, batched=True)
hf_val   = hf_val.map(tokenize_fn, batched=True)
hf_test  = hf_test.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=hf_tokenizer)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=6, id2label=id2label, label2id=label2id
)


In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'macro_f1': f1}

training_args = TrainingArguments(
    output_dir='./distilbert_emotion',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=100,
    report_to='none',
)

import inspect
trainer_kwargs = dict(
    model=bert_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_params = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_params:
    trainer_kwargs['processing_class'] = hf_tokenizer
else:
    trainer_kwargs['tokenizer'] = hf_tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()


## Stage 5 — Model Evaluation

In [ ]:
def evaluate_keras_model(model, X, y, name):
    probs = model.predict(X, batch_size=256, verbose=0)
    preds = np.argmax(probs, axis=1)
    acc = accuracy_score(y, preds)
    macro_f1 = f1_score(y, preds, average='macro')
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}')
    print(classification_report(y, preds, target_names=TARGET_EMOTIONS))
    cm = confusion_matrix(y, preds)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=TARGET_EMOTIONS, yticklabels=TARGET_EMOTIONS)
    plt.title(f'{name} — Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Pred')
    plt.show()
    return {'model': name, 'accuracy': acc, 'macro_f1': macro_f1}

results = []
results.append(evaluate_keras_model(lstm_model, X_test, y_test, 'LSTM'))
results.append(evaluate_keras_model(gru_model, X_test, y_test, 'GRU'))
results.append(evaluate_keras_model(bilstm_att_model, X_test, y_test, 'BiLSTM + Attention'))


In [ ]:
bert_preds_output = trainer.predict(hf_test)
bert_preds = np.argmax(bert_preds_output.predictions, axis=1)
bert_labels = bert_preds_output.label_ids

bert_acc = accuracy_score(bert_labels, bert_preds)
bert_f1 = f1_score(bert_labels, bert_preds, average='macro')
print(f'DistilBERT — Accuracy: {bert_acc:.4f} | Macro F1: {bert_f1:.4f}')
print(classification_report(bert_labels, bert_preds, target_names=TARGET_EMOTIONS))

cm = confusion_matrix(bert_labels, bert_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=TARGET_EMOTIONS, yticklabels=TARGET_EMOTIONS)
plt.title('DistilBERT — Confusion Matrix'); plt.ylabel('True'); plt.xlabel('Pred')
plt.show()

results.append({'model': 'DistilBERT', 'accuracy': bert_acc, 'macro_f1': bert_f1})


In [ ]:
comparison_df = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
comparison_df


In [ ]:
comparison_df.plot(x='model', y=['accuracy', 'macro_f1'], kind='bar', figsize=(7,4), rot=15)
plt.title('Model comparison'); plt.ylabel('score'); plt.ylim(0,1); plt.tight_layout(); plt.show()

best_model_name = comparison_df.iloc[0]['model']
print('Best model by macro F1:', best_model_name)


## Stage 6 — Attention Visualization
Visualize attention weights from the BiLSTM + Attention model to see which words
influenced each prediction.


In [ ]:
def visualize_attention(sentence, model_with_weights=bilstm_att_with_weights, max_len=MAX_LEN):
    clean = clean_text(sentence)
    seq = tokenizer.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    probs, weights = model_with_weights.predict(padded, verbose=0)
    probs, weights = probs[0], weights[0]

    tokens = clean.split()
    n = len(tokens)
    w = weights[:n]
    w = w / (w.sum() + 1e-9)

    pred_id = int(np.argmax(probs))
    print(f'Sentence: "{sentence}"')
    print(f'Predicted emotion: {id2label[pred_id]} (confidence {probs[pred_id]:.2%})')

    fig, ax = plt.subplots(figsize=(max(6, n*0.9), 1.6))
    ax.imshow(w[np.newaxis, :], cmap='Oranges', aspect='auto')
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=0)
    ax.set_yticks([])
    ax.set_title('Attention weights per token')
    plt.tight_layout(); plt.show()

    return dict(zip(tokens, w.tolist()))

_ = visualize_attention('I am extremely happy today.')
_ = visualize_attention('This makes me so angry and disgusted.')
_ = visualize_attention('I am terrified of what might happen next.')


## Stage 7 (part 1) — Export Artifacts for Deployment
Save everything needed to run the app locally in **VS Code**:
- Best Keras model (BiLSTM + Attention) **and** the fine-tuned DistilBERT
- Keras tokenizer (word index) as JSON
- Label map
- Model comparison table

Download the `deployment_artifacts` folder from Kaggle's Output panel after running,
then place it next to `app.py` in your VS Code project (see the deployment package).


In [ ]:
ART_DIR = '/kaggle/working/deployment_artifacts'
os.makedirs(ART_DIR, exist_ok=True)

bilstm_att_model.save(os.path.join(ART_DIR, 'bilstm_attention_model.keras'))

bert_model.save_pretrained(os.path.join(ART_DIR, 'distilbert_emotion'))
hf_tokenizer.save_pretrained(os.path.join(ART_DIR, 'distilbert_emotion'))

with open(os.path.join(ART_DIR, 'keras_tokenizer.json'), 'w') as f:
    f.write(tokenizer.to_json())

config = {
    'label2id': label2id,
    'id2label': id2label,
    'max_len': MAX_LEN,
    'embed_dim': EMBED_DIM,
    'vocab_size': vocab_size,
}
with open(os.path.join(ART_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

comparison_df.to_csv(os.path.join(ART_DIR, 'model_comparison.csv'), index=False)

print('Saved artifacts to', ART_DIR)
for root, dirs, files in os.walk(ART_DIR):
    for fn in files:
        print(os.path.join(root, fn))


### Next steps (Phase 2 & 3, per your workflow)
1. Download the `deployment_artifacts` folder from Kaggle (Output tab, top right → Download).
2. Move it into your VS Code project folder, next to `app.py` (see the separate
   Gradio deployment package I generated for you).
3. `pip install -r requirements.txt` and run `python app.py`.

By default the Gradio app loads the **DistilBERT** model (best accuracy). If you'd
rather deploy the custom BiLSTM+Attention model (to show live attention heatmaps),
set `MODEL_CHOICE = "bilstm_attention"` at the top of `app.py`.
